# Arbex C45: window30 T2 replication on 2025-05..2026-10 news

Pre-registered in `research/RESEARCH_TRACKER.md` §6 (C45, 2026-11-21) before any data for the period was
extracted. The extraction already ran in BigQuery (project `arbex-c45`, `research/sql/c45_gkg_extract.sql`).

**How to run:** Runtime > Run all. Two Google pop-ups ask for permission (Drive, then BigQuery); approve
them yourself. Cells 1-10 give the primary result (no GPU needed). Cell 11 (FinBERT, GPU) is optional and
only adds the reported-only T1/T3. Nothing here costs money: the project is a BigQuery sandbox.

In [ ]:
# 1. Settings
PROJECT = "arbex-c45"
BRANCH = "semantic-microservice"
COMMIT = "736ba8ea82c747dae91105a37dc5e412f8dbd88b"          # code pinned for the run (recorded in the outputs)
DRIVE_ROOT = "/content/drive/MyDrive/arbex_c45"

In [ ]:
# 2. Google Drive (keeps the shards, freeze, scores and HistData across a dropped session)
from google.colab import drive
drive.mount("/content/drive")
import os, pathlib
for sub in ("c45", "cache"):
    pathlib.Path(DRIVE_ROOT, sub).mkdir(parents=True, exist_ok=True)
print(os.listdir(DRIVE_ROOT))

In [ ]:
# 3. Code at COMMIT
import subprocess
if not os.path.exists("/content/Forex-Arbitrage"):
    subprocess.run(["git", "clone", "--branch", BRANCH, "https://github.com/KaikenK/Forex-Arbitrage.git",
                    "/content/Forex-Arbitrage"], check=True)
os.chdir("/content/Forex-Arbitrage")
subprocess.run(["git", "fetch", "-q", "origin", BRANCH], check=True)
subprocess.run(["git", "checkout", "-q", COMMIT], check=True)
CODE_COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code commit:", CODE_COMMIT)

In [ ]:
# 4. Dependencies (Colab already has pandas, numpy, scipy, torch, transformers, google-cloud-bigquery)
!pip -q install structlog yfinance pyarrow db-dtypes

In [ ]:
# 5. C45 outputs and the price cache live on Drive
repo_c45 = pathlib.Path("research/artifacts/c45")
if repo_c45.exists() and not repo_c45.is_symlink():
    import shutil; shutil.rmtree(repo_c45)
if not repo_c45.exists():
    repo_c45.symlink_to(pathlib.Path(DRIVE_ROOT, "c45"), target_is_directory=True)
os.environ["ARBEX_CACHE_DIR"] = str(pathlib.Path(DRIVE_ROOT, "cache"))
print(os.environ["ARBEX_CACHE_DIR"], "->", repo_c45.resolve())

In [ ]:
# 6. HistData M1 for the period: 2025 yearly file and January-October 2026 monthly files (owner approved)
!python -m research.fetch_histdata --pairs EUR/USD,GBP/USD,USD/JPY,USD/CHF --years 2025
!python -m research.fetch_histdata --pairs EUR/USD,GBP/USD,USD/JPY,USD/CHF --months 2026:1-10

In [ ]:
# 7. BigQuery sign-in (the arbex-c45 account)
from google.colab import auth
auth.authenticate_user()

In [ ]:
# 8. Ingest: daily counts + 18 monthly shards (reads the table with tabledata.list: no query quota)
!python -m research.run_c45 ingest --project arbex-c45

In [ ]:
# 9. Freeze (hash + missing days), written once
!python -m research.run_c45 freeze

In [ ]:
# 10. Dedup, then the primary test (T2-clock)
!python -m research.run_c45 score --stage dedup
!python -m research.run_c45 test

In [ ]:
# 11. Optional (GPU): FinBERT for the reported-only T1/T3, then the test again (T2 is unchanged)
import torch; print("GPU:", torch.cuda.is_available())
!python -m research.run_c45 score --stage finbert
!python -m research.run_c45 test